# Convert manufacturing CSV routes to XES event logs

Each CSV in `data/manufacturing/` contains process flow definitions (routes).
Each **route** becomes a **trace** (case).
Each **operation** within a route becomes an **event** — ordered by `ROUTEOPERORDER`.

UPT-level detail (unit process targets within each operation) is preserved as event attributes.

In [1]:
import pandas as pd
import pm4py
from pathlib import Path
import json

DATA_DIR = Path("../data/manufacturing")
OUTPUT_DIR = Path("../data/manufacturing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))
csv_files

[WindowsPath('../data/manufacturing/1VF_one_virtual_fab.csv'),
 WindowsPath('../data/manufacturing/legacy_system.csv'),
 WindowsPath('../data/manufacturing/tps_target-production-system.csv')]

## Converter Function
Define a function that reads a CSV, groups rows into events (one per operation per route), flattens UPT-level details into attributes, generates synthetic timestamps, and exports to XES.

In [2]:
def csv_to_xes(csv_path: Path) -> None:
    df = pd.read_csv(csv_path, dtype=str).fillna("")

    # ── columns common to all three CSVs ──
    route_cols = ["ROUTE", "ROUTERELATION", "FACILITY", "ROUTEID", "ROUTEDESCRIPTION"]
    upt_cols = ["UPT", "OPERATION_VARIANT", "OPERATIONDESCRIPTION",
                "SUMO", "DISPATCHFLAG", "UPTDESCRIPTION"]
    if "OPERUPTORDER" in df.columns:
        upt_cols.append("OPERUPTORDER")

    # ── ensure sortable numeric types for ordering columns ──
    for col in ["ROUTEOPERORDER", "SUMO", "OPERUPTORDER"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    order_cols = [c for c in ["OPERUPTORDER", "SUMO"] if c in df.columns]
    df = df.sort_values(["ROUTE", "ROUTEOPERORDER"] + order_cols)

    # ── group rows into events (one per operation per route) ──
    agg = {col: list for col in upt_cols}
    for col in route_cols:
        if col != "ROUTE":
            agg[col] = "first"

    events = df.groupby(["ROUTE", "ROUTEOPERORDER", "OPERATION"], as_index=False).agg(agg)
    events = events.sort_values(["ROUTE", "ROUTEOPERORDER"]).reset_index(drop=True)

    print(f"    → {len(events)} events across {events['ROUTE'].nunique()} routes")

    # ── flatten UPT lists into readable string attributes ──
    for src, dst in [("UPT", "upt_codes"),
                     ("OPERATION_VARIANT", "upt_variants"),
                     ("OPERATIONDESCRIPTION", "upt_descriptions"),
                     ("DISPATCHFLAG", "upt_dispatch_flags"),
                     ("SUMO", "upt_sumo"),
                     ("OPERUPTORDER", "upt_order")]:
        if src in events.columns:
            events[dst] = events[src].apply(lambda xs: ",".join(str(x) for x in xs))

    # ── synthetic timestamps (no real time data in CSVs) ──
    step_per_trace = events.groupby("ROUTE").cumcount()
    events["Timestamp"] = pd.Timestamp("2024-01-01") + pd.to_timedelta(step_per_trace * 2 + 8, unit="h")

    # ── PM4Py-standard column names ──
    events["case:concept:name"] = events["ROUTE"]
    events["concept:name"] = events["OPERATION"]
    events["time:timestamp"] = events["Timestamp"]

    keep = ["case:concept:name", "concept:name", "time:timestamp",
            "ROUTEOPERORDER"] + route_cols[1:] + [
        "upt_codes", "upt_variants", "upt_descriptions",
        "upt_dispatch_flags", "upt_sumo"]
    if "upt_order" in events.columns:
        keep.append("upt_order")
    events = events[keep].copy()

    # ── build event log ──
    log = pm4py.convert_to_event_log(events)

    out_path = OUTPUT_DIR / (csv_path.stem + ".xes")
    pm4py.write_xes(log, str(out_path))

    print(f"  ✓ {out_path.name}  ({len(log)} traces, {len(events)} events)")
    return log

## Process All CSVs
Iterate over each CSV file and convert it to XES.

In [3]:
for csv_file in csv_files:
    print(f"Processing {csv_file.name}...")
    csv_to_xes(csv_file)

Processing 1VF_one_virtual_fab.csv...
    → 30 events across 2 routes


c:\Users\safaya\Documents\GitHub\process-fragment-miner\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


exporting log, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

  ✓ 1VF_one_virtual_fab.xes  (2 traces, 30 events)
Processing legacy_system.csv...
    → 277 events across 3 routes


exporting log, completed traces ::   0%|          | 0/3 [00:00<?, ?it/s]

  ✓ legacy_system.xes  (3 traces, 277 events)
Processing tps_target-production-system.csv...
    → 570 events across 2 routes


exporting log, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

  ✓ tps_target-production-system.xes  (2 traces, 570 events)


## Sanity Check
Read back one of the generated XES files to verify the conversion worked correctly.

In [4]:
# quick sanity: inspect one converted log
log = pm4py.read_xes(str(OUTPUT_DIR / "1VF_one_virtual_fab.xes"), return_legacy_log_object=True)
print(f"Traces: {len(log)}")

for trace in list(log)[:2]:
    print(f"\n  Trace: {trace.attributes.get('concept:name')}")
    for ev in trace:
        print(f"    {ev.get('concept:name'):>8s} → UPTs: {ev.get('upt_codes', '')}")

parsing log, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

Traces: 2

  Trace: PREC5011
        1017 → UPTs: REALOT018
        1028 → UPTs: MOSFRA030
        1029 → UPTs: DESFRA020
        1033 → UPTs: MVPADR001
        1035 → UPTs: INMZZZ037
        2200 → UPTs: DIZZZZ055
        2230 → UPTs: WAFEXP004
        2250 → UPTs: DDCZZZ013,DISPZZ075,DDCZZZ013,CLOPCZ006,DISPZZ076,DDCZZZ013,DISPZZ077,IDFZZZ001,CLOPCZ006
        9020 → UPTs: DDGZZZ002,DISPZZ129,DDGZZZ002,INMZZZ035,DISPZZ130,DDGZZZ002,DISPZZ077,IDFZZZ001,CLOPCZ018
        9030 → UPTs: INMZZZ039
        9050 → UPTs: DDCZZZ035,DISPZZ075,DDCZZZ035,CLOPCZ006,DISPZZ076,DDCZZZ035,DISPZZ077,IDFZZZ001,CLOPCZ006
        9841 → UPTs: CHDPDP001
        9922 → UPTs: WALPRE001
        9924 → UPTs: DALPRE001
        9996 → UPTs: SHALOT015

  Trace: PREC5013
        1017 → UPTs: REALOT018
        1027 → UPTs: MOWFRA047
        2085 → UPTs: ERLRRZ001,ERLZZZ002,ERRZZZ001
        2095 → UPTs: INMZZZ041
        2200 → UPTs: DIZZZZ048
        2230 → UPTs: WAFEXP004
        2250 → UPTs: DDCZZZ013,DISPZZ075,

## Verify Output
List all generated XES files in the output directory.

In [5]:
# verify all three files were written
list(OUTPUT_DIR.glob("*.xes"))

[WindowsPath('../data/manufacturing/1VF_one_virtual_fab.xes'),
 WindowsPath('../data/manufacturing/legacy_system.xes'),
 WindowsPath('../data/manufacturing/tps_target-production-system.xes')]